# பாடம் 11 - முகவர்-முகவர் (A2A) நெறிமுறை


## அமைப்பு


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A நெறிமுறை என்றால் என்ன?

இந்த **Agent-to-Agent (A2A) நெறிமுறை** என்பது AI ஏஜென்ட்களை தொடர்பு கொள்ள, ஒருவரை மற்றொருவர் கண்டறியவும், மற்றும் ஒத்துழைக்கவும் அனுமதிக்கும் ஒரு திறந்த தரநிலையாகும் — அவை வெவ்வேறு கட்டமைப்புகளில் உருவாக்கப்பட்டிருந்தாலும் அல்லது
வெவ்வேறு சேவைகள் மூலம் ஹோஸ்ட் செய்யப்பட்டிருந்தாலும் கூட.

Key concepts:

- **கண்டறிதல்** – ஏஜென்ட்கள் தங்கள் திறன்களை விவரிக்கும் *Agent Card* ஒன்றை வெளியிடுகின்றனர், இது
  மற்ற ஏஜென்ட்கள் (அல்லது orchestrators) ஒரு பணிக்கான சரியான நிபுணரை எளிதில் கண்டுபிடிக்க உதவுகிறது.
- **செய்தி பரிமாற்றம்** – ஏஜென்ட்கள் ஒரு பொதுவான நெறிமுறையின் மூலம் கட்டமைக்கப்பட்ட செய்திகளை பரிமாறிக்கொள்கிறார்கள், அதனால் ஒரு
  ஏஜென்டில் இருந்து வரும் கோரிக்கையை மற்றொரு ஏஜென்டு அதன் உள்
அமலாக்கம் எப்படி இருந்தாலும் புரிந்து கொண்டு நிறைவேற்ற முடியும்.
- **பணி வாழ்க்கைச் சுழற்சி** – A2A *சமர்ப்பிக்கப்பட்டது*, *பணியில் உள்ளது*, *முடிக்கப்பட்டது*, மற்றும்
  *தோல்வியடைந்தது*, இது orchestrator-க்கு ஒப்படைக்கப்பட்ட பணி எவ்வாறு முன்னேறுகிறது என்பதில் முழு கண்ணோட்டத்தை வழங்குகிறது.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## சிறப்பு பயண முகவர்களை உருவாக்குதல்


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## பணிவழி மூலம் பல முகவர்கள் ஒத்துழைப்பு

நாங்கள் மூன்று முகவர்களை A2A செய்தி பரிமாற்றத்தை பிரதிபலிக்கும் தொடர் பணிவழியில் இணைக்கிறோம்:

1. **CurrencyExchangeAgent** பயனர் கோரிக்கையை பெறுகிறது மற்றும் நாணய வழிகாட்டுதலை வழங்குகிறது.
2. **ActivityPlannerAgent** செறிவூட்டப்பட்ட சூழலைப் பெறுகிறது மற்றும் செயல்பாடுகளுக்கான பரிந்துரைகளை சேர்க்கிறது.
3. **TravelManagerAgent** இரு உள்ளீடுகளையும் இறுதி பயண சுருக்கமாக ஒன்றிணைக்கிறது.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## உற்பத்தி சூழலில் A2A ஐப் புரிந்து கொள்வது

உற்பத்தி சூழலில் A2A புரோட்டோக்கால் பல்வேறு சேவைகள் இடையே சக்திவாய்ந்த பயன்பாட்டு நிலைகள் சாத்தியமாகின்றன:

| திறன் | விளக்கம் |
|---|---|
| **Cross-framework interop** | ஒரு ஃப்ரேம்வொர்க்கில் உருவாக்கப்பட்ட முகவர் (agent) மற்ற எந்த A2A-உடன் இணக்கமான ஃப்ரேம்வொர்க்கில் உருவாக்கப்பட்ட முகவருக்கும் பணிகளை ஒப்படுத்த முடியும்; இதனால் நிறுவனங்களை தாண்டிய உண்மையான இணக்கம் சாத்தியமாகிறது. |
| **Service boundaries** | முகவர்கள் தனித்த மைக்ரோசேவைகள், கிளவுட் பிரதேசங்கள் அல்லது வேறுபட்ட நிறுவனங்களில் இருந்தாலும் சீரான ஒத்துழைப்புடன் செயல்பட முடியும். |
| **Dynamic discovery** | ஒரு ஒர்கஸ்ட்ரேட்டர் இயக்கநேரத்தில் Agent Card பதிவேட்டைப் கேட்டு, கொடுக்கப்பட்ட துணைப் பணிக்கான சிறந்த நிபுணரைக் கண்டுபிடிக்க முடியும். |
| **Streaming & push notifications** | A2A நேரடி முன்னேற்றப் புதுப்பிப்புகளுக்காக Server-Sent Events (SSE)-ஐ மற்றும் நீண்டகாலமாக இயங்கும் பணிகளுக்கான புஷ் அறிவிப்புகளை ஆதரிக்கிறது. |

மேலே நாம் கட்டிய பணிச்சுழற்சி இந்த மாதிரியின் எளிமையாக்கப்பட்ட, செயல்முறை (in-process) பதிப்பாகும். உண்மையான டெப்ளாய்மெண்ட் நிலையில் ஒவ்வொரு முகவரும் ஒரு HTTP endpoint-ஐ வெளிப்படுத்தி, Agent Card ஒன்றை வெளியிட்டு, A2A JSON-RPC புரோட்டோக்கினூடாக தொடர்பு கொள்வார்.


## சுருக்கம்

இந்த பாடத்தில் நீங்கள் கற்றுக் கொண்டீர்கள்:

1. **A2A நெறிமுறை என்பது என்ன** — ஒரு திறந்த தரநிலை ஏஜென்ட்-இடையிலான கண்டுபிடிப்பு, செய்தியிடல்,
   மற்றும் பணிகள் நிர்வாகத்திற்காக.
2. **சிறப்பு ஏஜென்ட்களை எப்படி உருவாக்குவது** — ஒரு நாணய பரிமாற்ற ஏஜென்ட், ஒரு செயல்பாடு திட்டமிடுநர் ஏஜென்ட்,
   மற்றும் ஒரு பயண மேலாளர் ஒர்சஸ்ட்ரேட்டர்.
3. **ஏஜென்ட்களை ஒரு வேலைநெறியில் இணைப்பது எப்படி** — `WorkflowBuilder`-ஐ பயன்படுத்தி தொடர்
   செய்தி பரிமாற்றத்தை ஏஜென்ட்களுக்கிடையே மாடல் செய்ய.
4. **A2A உற்பத்தியில் எப்படி செயல்படுகிறது** — பல கட்டமைப்புகளுக்கும் பல சேவைகளுக்கும் இடையிலான ஒத்துழைப்பை
   டைனமிக் கண்டுபிடிப்பு மற்றும் ஸ்ட்ரீமிங் புதுப்பிப்புகளுடன் சாத்தியமாக்குகிறது.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
புறக்கணிப்பு:
இந்த ஆவணம் [Co-op Translator](https://github.com/Azure/co-op-translator) என்ற செயற்கை நுண்ணறிவு மொழிபெயர்ப்பு சேவையை பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்கிறோமெனினும், தானியங்கி மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறான தகவல்கள் இருக்கக்கூடும் என்பதை தயவுசெய்து கவனிக்கவும். அதன் மூல மொழியில் உள்ள ஆவணம் அதிகாரபூர்வ ஆதாரமாக கருதப்படவேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்முறை மனித மொழிபெயர்ப்பை பரிந்துரைக்கிறோம். இந்த மொழிபெயர்ப்பின் பயன்பாட்டினால் ஏற்படும் எந்தவொரு தவறான புரிதல்களுக்கும் அல்லது தவறான விளக்கங்களுக்கும் நாங்கள் பொறுப்பல்ல.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
